# 10 Audit pembagian data dan kebocoran resplit
Statistik pengulangan di dalam split resmi (Python dan Java), tumpang tindih antarsplit, dan breakdown BERTScore resplit
(contoh bocor vs bersih). Model resplit-nya sendiri dilatih di notebook 02 (varian full_resplit_bertsel). Menulis runs/split_audit.json.

## Setup
Pakai kernel Python dari ~/venvs/dq (venv yang menjalankan semua skrip). Sel ini menyamakan lingkungan dengan bg.sh:
cwd di codeqa_exp/code, pylib_metrics di path, Java untuk METEOR, dan GPU 0.

In [ ]:
import os, sys, json
from pathlib import Path

HOME = Path.home()
EXP = HOME / "inahuy/kacuy/codeqa_exp"
CODE = EXP / "code"
os.chdir(CODE)  # scripts import kc_common / registry from here
for p in (str(CODE), str(EXP / "pylib_metrics")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{EXP / 'pylib_metrics'}:{os.environ.get('PYTHONPATH', '')}"
os.environ["PATH"] = f"{HOME / 'envs/kc_tools/bin'}:{os.environ['PATH']}"  # Java for METEOR
os.environ["JAVA_HOME"] = str(HOME / "envs/kc_tools")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("cwd:", os.getcwd(), "| python:", sys.executable)

## split_audit.py

In [ ]:
VARIANTS = {
    'lengkap': [],
    'tanpa_bertscore': ['--no-bertscore'],
}
RUN = 'lengkap'  # ganti ke salah satu kunci VARIANTS
ARGS = VARIANTS[RUN]
sys.argv = ['split_audit.py', *ARGS]
print(RUN, ARGS)

**Sumber:** code/split_audit.py

```text
Split audit for CodeQA Python/Java + resplit leakage breakdown (BERTScore F1 primary).

Usage: python split_audit.py [--no-bertscore]
Writes runs/split_audit.json and prints a readable summary.
```

In [ ]:
import json, re, sys, statistics as st
from collections import Counter
from pathlib import Path

In [ ]:
ROOT = Path.home() / "inahuy/kacuy"
DATA = ROOT / "kc-codeqa-reproduce/data"
EXP = ROOT / "codeqa_exp"
SPLITS = ("train", "dev", "test")

In [ ]:
def rd(lang, split, ext):
    return (DATA / lang / split / f"{split}.{ext}").read_text(encoding="utf-8").splitlines()


def load(lang):
    out = {}
    for s in SPLITS:
        c, q, a = rd(lang, s, "code"), rd(lang, s, "question"), rd(lang, s, "answer")
        assert len(c) == len(q) == len(a), (lang, s)
        out[s] = [dict(code=x, question=y.strip(), answer=z.strip()) for x, y, z in zip(c, q, a)]
    return out

In [ ]:
norm = lambda code: " ".join(code.split())
toks = lambda s: s.split()

In [ ]:
def qtype(q):
    w = q.lower().split()
    return " ".join(w[:2]) if w else ""

In [ ]:
def split_stats(rows):
    codes = Counter(norm(r["code"]) for r in rows)
    yn = sum(r["answer"].lower() in ("yes", "no") for r in rows)
    ctoks = [set(toks(r["code"].lower())) for r in rows]
    # answer fully contained in code tokens = "extractive" answer
    extr = sum(all(t in ct for t in toks(r["answer"].lower())) for r, ct in zip(rows, ctoks))
    qa = Counter((r["question"], r["answer"]) for r in rows)
    ans = Counter(r["answer"] for r in rows)
    return {
        # share of rows whose (q,a) / answer occurs more than once inside the same split
        "qa_pair_repeated_within_pct": round(100 * sum(qa[(r["question"], r["answer"])] > 1 for r in rows) / len(rows), 2),
        "answer_repeated_within_pct": round(100 * sum(ans[r["answer"]] > 1 for r in rows) / len(rows), 2),
        "n": len(rows),
        "unique_codes": len(codes),
        "qa_per_code_mean": round(len(rows) / len(codes), 2),
        "code_tokens_median": st.median(len(toks(r["code"])) for r in rows),
        "question_tokens_mean": round(st.mean(len(toks(r["question"])) for r in rows), 2),
        "answer_tokens_mean": round(st.mean(len(toks(r["answer"])) for r in rows), 2),
        "yes_no_pct": round(100 * yn / len(rows), 2),
        "answer_extractive_pct": round(100 * extr / len(rows), 2),
        "top_qtypes": [(k, round(100 * v / len(rows), 1)) for k, v in Counter(qtype(r["question"]) for r in rows).most_common(6)],
    }

In [ ]:
def overlap(target, train):
    tc = {norm(r["code"]) for r in train}
    tq = {r["question"] for r in train}
    tqa = {(r["question"], r["answer"]) for r in train}
    tcqa = {(norm(r["code"]), r["question"]) for r in train}
    ta = {r["answer"] for r in train}
    n = len(target)
    pct = lambda k: round(100 * k / n, 2)
    return {
        "code_seen": pct(sum(norm(r["code"]) in tc for r in target)),
        "question_seen": pct(sum(r["question"] in tq for r in target)),
        "qa_pair_seen": pct(sum((r["question"], r["answer"]) in tqa for r in target)),
        "code_and_question_seen": pct(sum((norm(r["code"]), r["question"]) in tcqa for r in target)),
        "answer_string_seen": pct(sum(r["answer"] in ta for r in target)),
    }

In [ ]:
def flags_vs_train(target, train):
    """Per-example leakage flags for the resplit test."""
    code2ans = {}
    for r in train:
        code2ans.setdefault(norm(r["code"]), set()).add(r["answer"])
    tcq = {(norm(r["code"]), r["question"]) for r in train}
    tqa = {(r["question"], r["answer"]) for r in train}
    ta = {r["answer"] for r in train}
    out = []
    for r in target:
        k = norm(r["code"])
        out.append({
            "qa_seen": (r["question"], r["answer"]) in tqa,
            "answer_seen": r["answer"] in ta,
            "code_seen": k in code2ans,
            "code_q_seen": (k, r["question"]) in tcq,
            "same_answer_for_code_in_train": r["answer"] in code2ans.get(k, ()),
        })
    return out

In [ ]:
def main():
    use_bs = "--no-bertscore" not in sys.argv
    report = {}
    for lang in ("python", "java"):
        d = load(lang)
        report[lang] = {
            "stats": {s: split_stats(d[s]) for s in SPLITS},
            "overlap_vs_train": {s: overlap(d[s], d["train"]) for s in ("dev", "test")},
            "overlap_dev_test": overlap(d["test"], d["dev"]),
        }
        print("==", lang)
        print(json.dumps(report[lang], indent=1))

    # --- resplit: rebuild new splits from the stored map ("split:idx" refs)
    py = load("python")
    m = json.loads((EXP / "maps/resplit_seed42.json").read_text())
    get = lambda ref: py[ref.split(":")[0]][int(ref.split(":")[1])]
    new = {s: [get(r) for r in m[s]] for s in SPLITS}
    rs = {
        "stats": {s: split_stats(new[s]) for s in SPLITS},
        "overlap_vs_train": {s: overlap(new[s], new["train"]) for s in ("dev", "test")},
        "test_origin": dict(Counter(r.split(":")[0] for r in m["test"])),
    }

    # --- per-example scores of the resplit run, split by leakage flag
    preds = [json.loads(l) for l in (EXP / "runs/codet5_full_resplit_bertsel/test_predictions.jsonl").open()]
    pool = py["train"] + py["dev"] + py["test"]  # pred id = pool index (codet5_p.py --resplit)
    test_ids_from_map = [ {"train": 0, "dev": 56085, "test": 63085}[r.split(":")[0]] + int(r.split(":")[1]) for r in m["test"]]
    assert sorted(p["id"] for p in preds) == sorted(test_ids_from_map), "pred ids do not match resplit map"
    for p in preds:  # sanity: reference matches pooled answer
        assert p["reference"].strip() == pool[p["id"]]["answer"], p["id"]
    fl = flags_vs_train([pool[p["id"]] for p in preds], new["train"])
    id2origin = {i: r.split(":")[0] for i, r in zip(test_ids_from_map, m["test"])}

    orig = [json.loads(l) for l in (EXP / "runs/codet5_full_bertsel/test_predictions.jsonl").open()]
    if use_bs:
        from bert_score import score
        cands = [p["prediction"] for p in preds] + [p["prediction"] for p in orig]
        refs = [p["reference"] for p in preds] + [p["reference"] for p in orig]
        _, _, F = score(cands, refs, model_type="roberta-large", lang="en", batch_size=128, device="cuda", verbose=False)
        F = (100 * F).tolist()
        for p, f in zip(preds + orig, F):
            p["BERTScore_F1"] = f

    def agg(items):
        if not items:
            return {"n": 0}
        o = {"n": len(items), "token_F1": round(st.mean(p["F1"] for p in items), 2),
             "EM": round(st.mean(p["EM"] for p in items), 2)}
        if use_bs:
            o["BERTScore_F1"] = round(st.mean(p["BERTScore_F1"] for p in items), 2)
        return o

    buckets = {
        "all": [True] * len(preds),
        "code_seen_in_train": [f["code_seen"] for f in fl],
        "code_unseen": [not f["code_seen"] for f in fl],
        "code_seen_same_answer_in_train": [f["same_answer_for_code_in_train"] for f in fl],
        "code_seen_new_answer": [f["code_seen"] and not f["same_answer_for_code_in_train"] for f in fl],
        "code_and_question_seen": [f["code_q_seen"] for f in fl],
        "qa_pair_seen": [f["qa_seen"] for f in fl],
        "answer_seen": [f["answer_seen"] for f in fl],
        "clean_no_code_no_answer_seen": [not f["code_seen"] and not f["answer_seen"] for f in fl],
        "origin_train": [id2origin[p["id"]] == "train" for p in preds],
        "origin_dev": [id2origin[p["id"]] == "dev" for p in preds],
        "origin_test": [id2origin[p["id"]] == "test" for p in preds],
        "origin_test_clean": [id2origin[p["id"]] == "test" and not f["code_seen"] and not f["answer_seen"] for p, f in zip(preds, fl)],
    }
    rs["resplit_test_scores"] = {k: agg([p for p, b in zip(preds, v) if b]) for k, v in buckets.items()}
    # original split, same flags vs the original train, for a like-for-like comparison
    assert all(p["reference"].strip() == py["test"][p["id"]]["answer"] for p in orig), "orig pred ids != test index"
    ofl = flags_vs_train([py["test"][p["id"]] for p in orig], py["train"])
    rs["original_test_scores"] = {
        "all": agg(orig),
        "answer_seen": agg([p for p, f in zip(orig, ofl) if f["answer_seen"]]),
        "answer_unseen": agg([p for p, f in zip(orig, ofl) if not f["answer_seen"]]),
        "qa_pair_seen": agg([p for p, f in zip(orig, ofl) if f["qa_seen"]]),
    }
    report["python_resplit_seed42"] = rs
    print("== python resplit seed42")
    print(json.dumps(rs, indent=1))
    (EXP / "runs/split_audit.json").write_text(json.dumps(report, indent=2))
    print("WROTE", EXP / "runs/split_audit.json")

In [ ]:
if __name__ == "__main__":
    main()